# 📊 InsightX AI: Cloud-Ready Exploratory Data Analysis (EDA)
### Hybrid Cloud Architecture: Kaggle / Google Colab Starter

This notebook is pre-configured to run seamlessly across:
1. **Google Colab** (via GitHub clone + KaggleHub / Google Drive)
2. **Kaggle Notebooks** (via direct native dataset mount at `/kaggle/input/brazilian-ecommerce`)
3. **Local Workstation** (via `data/raw` if lightweight testing is desired)

---

## 1. Environment Detection and Repository Sync
Detects if running on Colab or Kaggle, clones your GitHub repository, and installs `insightx`.

In [ ]:
import sys
import os
from pathlib import Path

# Detect runtime
IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or Path('/kaggle/input').exists()

env_name = 'Google Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local')
print(f"Detected Runtime Environment: {env_name}")

# If on cloud, pull latest code from GitHub
if IN_COLAB or IN_KAGGLE:
    # Configured for GitHub repository
    GITHUB_USERNAME = "namrata-dev1"
    REPO_URL = f"https://github.com/{GITHUB_USERNAME}/InsightX-AI.git"
    
    if not Path("src/insightx").exists():
        if not Path("InsightX-AI").exists():
            print(f"Cloning {REPO_URL}...")
            !git clone {REPO_URL}
            %cd InsightX-AI
        else:
            %cd InsightX-AI
    
    # Pull latest updates and install editable package
    !git pull
    !pip install -e . -q
    print("✅ InsightX AI package successfully linked and loaded!")

## 2. Cloud Storage and Data Setup
* **Colab**: Mounts Google Drive (for saving model weights/plots) and uses `kagglehub` to load Olist directly into RAM.
* **Kaggle**: Dataset is already natively mounted at `/kaggle/input/brazilian-ecommerce` (zero setup needed).

In [ ]:
if IN_COLAB:
    # 1. Mount Google Drive to store model artifacts and reports
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"Drive mount skipped: {e}")

    # 2. Fast direct download into Colab memory via kagglehub
    if not Path('/content/data').exists():
        print("Downloading Olist dataset directly to Colab...")
        !pip install -q kagglehub
        import kagglehub
        downloaded_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
        print(f"✅ Dataset cached at: {downloaded_path}")
        os.environ["OLIST_DATA_DIR"] = downloaded_path

## 3. Verify Environment Path Resolver
Our modular `insightx.config` automatically points to the correct dataset and artifact paths without hardcoding.

In [ ]:
from insightx.config import get_environment_name, resolve_data_dir, resolve_artifacts_dir
from insightx.data.loader import OlistDataLoader

print(f"Active Environment:      {get_environment_name()}")
print(f"Resolved Data Directory: {resolve_data_dir()}")
print(f"Resolved Artifacts Path: {resolve_artifacts_dir()}")

# Initialize loader
loader = OlistDataLoader()

## 4. Olist Dataset Structural Summary
Inspect all 9 tables (file size, rows, columns, and availability status) efficiently.

In [ ]:
summary_df = loader.inspect_dataset_summary()
summary_df

## 5. Sample Inspection: Fact Tables
Inspect the primary orders table and verify the `customer_id` vs `customer_unique_id` relationship.

In [ ]:
# Load 1,000 order sample for instant validation
orders_df = loader.load_table("olist_orders_dataset.csv", nrows=1000)
customers_df = loader.load_table("olist_customers_dataset.csv", nrows=1000)

print("Orders Sample Shape:   ", orders_df.shape)
print("Customers Sample Shape:", customers_df.shape)
print("Unique individual buyers in sample:", customers_df['customer_unique_id'].nunique())
print("Total transaction IDs in sample:   ", customers_df['customer_id'].nunique())

display(orders_df.head(3))
display(customers_df.head(3))